# Advanced RAGハンズオン

## データセットの登録

In [ ]:
import json
import os

import weave
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

# qa-advance.jsonl を Weave の名前付きデータセットとして登録する
# 内容が変わると Weave が新しいバージョンを切り、同じ内容なら同じバージョンのまま
weave.init(os.environ["WANDB_PROJECT"])
with open("../data/dataset/qa-advance.jsonl") as f:
    rows = [json.loads(line) for line in f if line.strip()]
ref = weave.publish(weave.Dataset(name="qa-advance", rows=rows))  # type: ignore
print(f"qa-advance:{ref.digest}  ({len(rows)} rows)")

## 評価関数（スコアラー）の実装

### 目的の文書が検索にヒットしているか判定（retrieival_hit）

In [ ]:
import re

PAGE_RE = re.compile(r"page(\d+)")
PAGE_RANGE_RE = re.compile(r"page(\d+)-(\d+)(?:#|$)")


def covers(doc_id: str, source_id: str) -> bool:
    """検索結果のチャンク doc_id が、正解の source_id に当たるか。

    `相対パス#シート名` は、そのシートのどのチャンク(`#1`, `#2`, ...)でも当たり。
    `相対パス#page18` は、ページ範囲のチャンク(`#page18-21`、`#page15-18#2` など)が 18 を含めば当たり。
    """
    if doc_id == source_id or doc_id.startswith(source_id + "#"):
        return True
    path, _, unit = source_id.rpartition("#")
    page = PAGE_RE.fullmatch(unit)
    if not page or not doc_id.startswith(path + "#"):
        return False
    span = PAGE_RANGE_RE.match(doc_id, len(path) + 1)
    return bool(span) and int(span[1]) <= int(page[1]) <= int(span[2])


def rank_of(source_id: str, ranked_ids: list[str]) -> int | None:
    """source_id に当たるチャンクの最上位の順位(1 始まり)。ranked_ids に無ければ None。"""
    return next((n for n, i in enumerate(ranked_ids, 1) if covers(i, source_id)), None)


@weave.op
def retrieval_hit(source_ids: list[list[str]], output: dict) -> dict:
    """正解の文書の組(source_ids)のどれか 1 組が、検索結果に全部入っているか。"""
    ids = [c["id"] for c in output["contexts"]]
    recalls = []
    for group in source_ids:  # どれか 1 組が揃えば当たり
        found = [any(covers(i, s) for i in ids) for s in group]
        recalls.append(sum(found) / len(found))
    ranks = [[rank_of(s, output["ranked_ids"]) for s in group] for group in source_ids]
    needs = [max(r) for r in ranks if None not in r]
    return {
        # どれか 1 組の正解文書が、LLM に渡したチャンク(contexts)に全部入ったか
        "hit": max(recalls) == 1,
        # 組ごとに「正解文書のうち contexts に入った割合」を出し、その最大値
        "recall": max(recalls),
        # 組ごと・正解文書ごとの、ranked_ids での順位。ranked_ids に無ければ None
        "ranks": ranks,
        # ranked_ids の上位何件まで取れば、どれか 1 組が揃うか。どの組も揃わなければ None
        "needed_k": min(needs, default=None),
    }

### 回答内容が正しいか判定（answer_correct）

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI()

JUDGE_PROMPT = (
    "質問に対する回答を採点します。回答が正解と同じ内容を含んでいれば correct を true に、"
    "誤りや欠落があれば false にしてください。表現の違いは問いません。"
)


class Judgement(BaseModel):
    """判定の形。Structured Outputs で、必ずこの形で返ってくる。"""

    correct: bool
    reason: str = Field(description="理由を一文で")


@weave.op
def answer_correct(question: str, expected: str, output: dict) -> dict:
    """回答が正解(expected)と同じ内容か、LLM に判定させる。"""
    res = client.chat.completions.parse(
        model="gpt-6-luna",
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {
                "role": "user",
                "content": f"# 質問\n{question}\n\n# 正解\n{expected}\n\n# 回答\n{output['answer']}",
            },
        ],
        response_format=Judgement,
        reasoning_effort="low",
    )
    return res.choices[0].message.parsed.model_dump()

## 段階1：シンプルなRAG

### 段階1の評価の実行

In [ ]:
import sys

import lancedb

sys.path.append("..")
from app.advanced_rag.rag import DB_DIR, PROMPTS, RagModel, table_name  # noqa: E402

In [ ]:
# 段階1: basic 抽出器のインデックスで、ほかは RagModel の既定のまま
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階1",
)
await evaluation.evaluate(RagModel(index="basic"))

## 段階2：検索対象を絞り込む

### 段階2の評価の実行

In [ ]:
# 段階2: 段階1 に加えて、検索対象を質問の資料群(collection)に絞り込む
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階2",
)
await evaluation.evaluate(RagModel(index="basic", use_collection=True))

### 分析:チャンクの中身を確認する（q15）

In [ ]:
def show_chunks(doc_id: str, index: str = "basic") -> None:
    """文書 ID に当たるチャンクの、登録されているテキストを出す。

    doc_id は検索結果の id でも、データセットの source_ids の値でもよい(当たりの判定は covers と同じ)。
    """
    table = lancedb.connect(DB_DIR).open_table(table_name(index))
    rows = table.to_arrow().select(["id", "title", "text"]).to_pylist()
    hits = [r for r in rows if covers(r["id"], doc_id)]
    if not hits:
        print(f"インデックス {index} に {doc_id} に当たるチャンクはありません")
    for r in hits:
        print("-" * 80 + f"\n{r['id']}\n" + "-" * 80 + f"\n{r['title']}\n{r['text']}\n")


# q15「オートフィルターで2つの列を使って絞り込む手順は?」でヒットして欲しいページ
show_chunks("LibreOffice研修テキスト/LibreOffice_Introduction#page97")
show_chunks("LibreOffice研修テキスト/LibreOffice_Introduction#page98")
show_chunks("LibreOffice研修テキスト/LibreOffice_Introduction#page99")

### 分析:チャンクの中身を確認する（q6）

In [ ]:
show_chunks(
    "システム開発/B1_顧客管理システム/030_アプリ設計/030_インタフェース設計/外部インタフェース設計書_B10101P_顧客検索応答電文_(JSON)#1. 外部インタフェース仕様"
)

## 段階3：文書に合わせた情報抽出


### 分析:チャンクの中身を確認する（q15）


In [ ]:
# 同じページを basic と structured で比べる
id = "LibreOffice研修テキスト/LibreOffice_Introduction#page97"
print("段階2")
show_chunks(id, index="basic")
print("段階3")
show_chunks(id, index="structured")

### 分析:チャンクの中身を確認する（q6）

In [ ]:
# 同じページを basic と structured で比べる
id = "システム開発/B1_顧客管理システム/030_アプリ設計/030_インタフェース設計/外部インタフェース設計書_B10101P_顧客検索応答電文_(JSON)#1. 外部インタフェース仕様"
print("段階2")
show_chunks(id, index="basic")
print("段階3")
show_chunks(id, index="structured")

### 段階3の評価の実行

In [ ]:
# 段階3: 段階2 に加えて、文書の形式に合わせて情報を抜き出した structured 抽出器のインデックスを使う
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階3",
)
await evaluation.evaluate(RagModel(index="structured", use_collection=True))

### 分析:チャンクの中身を確認する（q5）

In [ ]:
show_chunks(
    "システム開発/A1_プロジェクト管理システム/010_要件定義/040_画面設計/画面遷移図_A1_プロジェクト管理システム#2. プロジェクト管理(A102)",
    index="structured",
)

### 分析:チャンクの中身を確認する（q18)

In [ ]:
show_chunks(
    "厚労省資料/健康日本21（第二次）参考資料スライド集#slide36", index="structured"
)

## 段階4：マルチモーダルLLMの活用

### 分析:チャンクの中身を確認する（q6）

In [ ]:
# 同じページを structured と vision で比べる
id = "システム開発/B1_顧客管理システム/030_アプリ設計/030_インタフェース設計/外部インタフェース設計書_B10101P_顧客検索応答電文_(JSON)#1. 外部インタフェース仕様"
print("段階3")
show_chunks(id, index="structured")
print("段階4")
show_chunks(id, index="vision")

### 分析:チャンクの中身を確認する（q5）

In [ ]:
# 同じページを structured と vision で比べる
id = "システム開発/A1_プロジェクト管理システム/010_要件定義/040_画面設計/画面遷移図_A1_プロジェクト管理システム#2. プロジェクト管理(A102)"
print("段階3")
show_chunks(id, index="structured")
print("段階4")
show_chunks(id, index="vision")

### 分析:チャンクの中身を確認する（q18）

In [ ]:
# 同じページを structured と vision で比べる
id = "厚労省資料/健康日本21（第二次）参考資料スライド集#slide36"
print("段階3")
show_chunks(id, index="structured")
print("段階4")
show_chunks(id, index="vision")

### 段階4の評価の実行

In [ ]:
# 段階4: 段階3 に加えて、図や画像の説明をマルチモーダル LLM で足した vision 抽出器のインデックスを使う
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階4",
)
await evaluation.evaluate(RagModel(index="vision", use_collection=True))

## 段階5：ハイブリッド検索

### 段階5の評価の実行

In [ ]:
# 段階5: 段階4 に加えて、ベクトル検索と全文検索(BM25)の結果を RRF で混ぜるハイブリッド検索を使う
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階5",
)
await evaluation.evaluate(
    RagModel(index="vision", use_collection=True, search="hybrid")
)

## 段階6：リランキング

### 段階6の評価の実行

In [ ]:
# 段階6: 段階5 に加えて、検索の上位 20 件(rerank_depth)を LLM が 0〜10 点で採点し、点の順に並べ直す
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階6",
)
await evaluation.evaluate(
    RagModel(index="vision", use_collection=True, search="hybrid", rerank="llm")
)

## 段階7：プロンプトのルーティング

### 段階7の評価の実行

In [ ]:
# 段階7: 段階6 に加えて、回答用の system prompt に規則(RULES_PROMPT)を足し、質問の資料群(collection)ごとの説明も足す
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階7",
)
await evaluation.evaluate(
    RagModel(
        index="vision",
        use_collection=True,
        search="hybrid",
        rerank="llm",
        **PROMPTS["collection"],
    )
)

## 段階8：検索後のチャンクの再構成

### 段階8の評価の実行

In [ ]:
# 段階8: 段階7 に加えて、並べ直しの点が閾値(score_threshold)以上のチャンクだけを選び、
# 選んだチャンクと同じ区切りのほかのチャンクと、手順の途中から始まる区切りの手前のチャンクも渡す
evaluation = weave.Evaluation(
    name="eval-qa-advance",
    dataset=weave.ref("qa-advance").get(),
    scorers=[retrieval_hit, answer_correct],
    evaluation_name="段階8",
)
await evaluation.evaluate(
    RagModel(
        index="vision",
        use_collection=True,
        search="hybrid",
        rerank="llm",
        select="score",
        expand="unit+steps",
        **PROMPTS["collection"],
    )
)